<a href="https://colab.research.google.com/github/AdwitaKumari12/Online-Shopping-Delivery-Logistics-Analysis/blob/main/Delivery_Logistic_Adwita_Kumari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Importing and using the given Dataset for analyzing***

In [1]:
# Import necessary libraries
import pandas as pd   # Loading the panda libraries using standard 'pd' alias for working tabular data.
# Load the dataset into a pandas DataFrame to inspect the data and analyze it.
df = pd.read_csv("/content/Delivery_Logistics.csv")

print("DELIVERY LOGISTIC DATABASE")
print(df.head())

print("/nShape: ")
print(df.shape)

print("/columns: ")
print(df.columns)

DELIVERY LOGISTIC DATABASE
   delivery_id delivery_partner      package_type vehicle_type delivery_mode  \
0       250.99        delhivery  automobile parts         bike      same day   
1       250.99       xpressbees         cosmetics       ev van       express   
2       250.99        shadowfax         groceries        truck       two day   
3       250.99              dhl       electronics       ev van      same day   
4       250.99              dhl          clothing          van       two day   

    region weather_condition  distance_km  package_weight_kg  \
0     west             clear        297.0              46.96   
1  central              cold         89.6              47.39   
2     east             rainy        273.5              26.89   
3     east              cold        269.7              12.69   
4    north             foggy        256.7              37.02   

             delivery_time_hours            expected_time_hours delayed  \
0  1970-01-01 00:00:00.000000008

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


***Finding out the Dataset shape and Missing Values with Duplicate ID records***

In [3]:
# Load dataset and inspect shape, columns, and data types
df = pd.read_csv("Delivery_Logistics.csv")
print("Shape:", df.shape, "| Columns:", list(df.columns))
print("\nData Types:\n", df.dtypes)

# Finding out missing values and duplicate ID records
print("\nMissing Values:\n", df.isnull().sum())
id_col = "Shipment_ID" if "Shipment_ID" in df.columns else "delivery_id"
print(f"Duplicate '{id_col}' count:", df.duplicated(subset=[id_col]).sum())

Shape: (25000, 15) | Columns: ['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'distance_km', 'package_weight_kg', 'delivery_time_hours', 'expected_time_hours', 'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost']

Data Types:
 delivery_id            float64
delivery_partner        object
package_type            object
vehicle_type            object
delivery_mode           object
region                  object
weather_condition       object
distance_km            float64
package_weight_kg      float64
delivery_time_hours     object
expected_time_hours     object
delayed                 object
delivery_status         object
delivery_rating          int64
delivery_cost          float64
dtype: object

Missing Values:
 delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km         

***Filling out the missing values and converting dates and time***

In [4]:
# 3. Filling missing cost/time values using median/mode
cost_col = "Shipping_Cost" if "Shipping_Cost" in df.columns else "delivery_cost"
df[cost_col] = df[cost_col].fillna(df[cost_col].median())
df["delivery_time_hours"] = df["delivery_time_hours"].fillna(
    df["delivery_time_hours"].mode()[0]
)

# Converting dates/times to datetime and compute Delivery_Days
if "Order_Date" in df.columns and "Delivery_Date" in df.columns:
  df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
  df["Delivery_Date"] = pd.to_datetime(df["Delivery_Date"], errors="coerce")
  df["Delivery_Days"] = (df["Delivery_Date"] - df["Order_Date"]).dt.days
else:
  df["delivery_time_hours"] = pd.to_datetime(
      df["delivery_time_hours"], errors="coerce"
  )
  hours = (
      df["delivery_time_hours"]
      .astype(str)
      .str.split(".")
      .str[-1]
      .astype(float)
  )
  df["Delivery_Days"] = hours / 24.0

***Calculating Success Rate and Average Delivery Duration***

In [5]:
# Displaying total shipments, delivered shipments, and delivery success %
total = len(df)
delivered = (df["delivery_status"].str.lower() == "delivered").sum()
print(
    f"Total: {total} | Delivered: {delivered} | Success:"
    f" {delivered / total * 100:.2f}%"
)

# Displaying Average delivery duration by shipping method
print("\nAvg Delivery_Days by Shipping Method:")
print(df.groupby("delivery_mode")["Delivery_Days"].mean().round(4))

Total: 25000 | Delivered: 18331 | Success: 73.32%

Avg Delivery_Days by Shipping Method:
delivery_mode
express     0.2596
same day    0.2602
standard    0.2599
two day     0.2616
Name: Delivery_Days, dtype: float64


***Partner Ranking***

In [6]:
# Displaying rank delivery partners by shipment count and delivery duration
partner_rank = df.groupby("delivery_partner").agg(
    shipments=(id_col, "count"), avg_days=("Delivery_Days", "mean")
)
partner_rank["rank_by_count"] = partner_rank["shipments"].rank(ascending=False)
partner_rank["rank_by_speed"] = partner_rank["avg_days"].rank(ascending=True)
print("\nPartner Rankings:\n", partner_rank.round(4))

# Calculating and displaying the delivery success rate across shipping methods
mode_success = (
    df.groupby("delivery_mode")["delivery_status"]
    .apply(lambda s: (s.str.lower() == "delivered").mean() * 100)
    .round(2)
)
print("\nSuccess Rate (%) by Shipping Method:\n", mode_success)

# Printing ou the Top 10 shipments with longest delivery duration
print("\nTop 10 Longest Shipments:")
print(
    df.nlargest(10, "Delivery_Days")[
        [id_col, "delivery_partner", "delivery_mode", "Delivery_Days"]
    ]
)


Partner Rankings:
                   shipments  avg_days  rank_by_count  rank_by_speed
delivery_partner                                                   
amazon logistics       2711    0.2616            9.0            7.0
blue dart              2798    0.2623            5.0            8.0
delhivery              2786    0.2565            6.0            2.0
dhl                    2802    0.2604            3.0            4.0
ecom express           2722    0.2603            8.0            3.0
ekart                  2801    0.2612            4.0            5.0
fedex                  2818    0.2559            2.0            1.0
shadowfax              2736    0.2613            7.0            6.0
xpressbees             2826    0.2635            1.0            9.0

Success Rate (%) by Shipping Method:
 delivery_mode
express      26.22
same day     67.46
standard    100.00
two day      99.57
Name: delivery_status, dtype: float64

Top 10 Longest Shipments:
       delivery_id  delivery_partner d

***Day Label Counts***

In [7]:
# Printing the dictionary of delivery counts by status & set of unique delivery partners
status_counts = {}
for st in df["delivery_status"]:
  status_counts[st] = status_counts.get(st, 0) + 1

unique_partners = set(df["delivery_partner"].dropna())
print("Status Dictionary:", status_counts)
print("Unique Partners:", unique_partners)


# Using Function to label shipments using day limits
def label_shipment(days):
  if days <= 0.25:
    return "On-Time"
  elif days <= 0.50:
    return "Slightly Delayed"
  return "Highly Delayed"


df["delay_label"] = df["Delivery_Days"].apply(label_shipment)
print("\nDelay Label Counts:\n", df["delay_label"].value_counts())

Status Dictionary: {'delivered': 18331, 'delayed': 5341, 'failed': 1328}
Unique Partners: {'blue dart', 'fedex', 'dhl', 'delhivery', 'ekart', 'ecom express', 'shadowfax', 'amazon logistics', 'xpressbees'}

Delay Label Counts:
 delay_label
On-Time             13938
Slightly Delayed    10241
Highly Delayed        821
Name: count, dtype: int64


***Calculating Mean and Median***

In [8]:
import numpy as np

# Calculating the NumPy mean, median, and standard deviation of Delivery_Days
arr = df["Delivery_Days"].dropna().to_numpy()
print(
    f"Mean: {np.mean(arr):.4f} | Median: {np.median(arr):.4f} | Std:"
    f" {np.std(arr):.4f}"
)

# Displaying IQR outliers and monthly/regional performance comparison
q1, q3 = np.percentile(arr, [25, 75])
upper = q3 + 1.5 * (q3 - q1)
print("IQR outliers count:", len(df[df["Delivery_Days"] > upper]))
if "Order_Date" in df.columns:
  print(
      "By month:\n",
      df.groupby(df["Order_Date"].dt.to_period("M"))["Delivery_Days"].mean(),
  )
else:
  print(
      "No Month/Order_Date in CSV; by region:\n",
      df.groupby("region")[["Delivery_Days", "delivery_rating"]].mean().round(3),
  )

Mean: 0.2603 | Median: 0.2500 | Std: 0.1309
IQR outliers count: 450
No Month/Order_Date in CSV; by region:
          Delivery_Days  delivery_rating
region                                 
central          0.262            3.661
east             0.259            3.682
north            0.260            3.680
south            0.258            3.668
west             0.262            3.640


***Reccomendations***

In [9]:
# Using Simple Shipment class and concluding with five improvement recommendations
class Shipment:
  def __init__(self, shipment_id, partner, delivery_days):
    self.shipment_id, self.partner, self.delivery_days = shipment_id, partner, delivery_days
  def show(self):
    return f"ID={self.shipment_id}, Partner={self.partner}, Days={self.delivery_days:.3f}"
#loc
r = df.iloc[0]
print("\nSample:", Shipment(r[id_col], r["delivery_partner"], r["Delivery_Days"]).show())

recs = [
    "1. Audit IQR outlier and top-10 longest shipments for route delays.",
    "2. Favor faster, higher-success partners during peak load.",
    "3. Tighten dispatch cutoffs for 'express' and 'same day' modes.",
    "4. Add weather/region buffers for slower or lower-rated routes.",
    "5. Include real Order_Date and Delivery_Date columns for monthly tracking.",
]
print("\nRecommendations:\n" + "\n".join(recs))


Sample: ID=250.99, Partner=delhivery, Days=0.333

Recommendations:
1. Audit IQR outlier and top-10 longest shipments for route delays.
2. Favor faster, higher-success partners during peak load.
3. Tighten dispatch cutoffs for 'express' and 'same day' modes.
4. Add weather/region buffers for slower or lower-rated routes.
5. Include real Order_Date and Delivery_Date columns for monthly tracking.


***File Handling***

In [10]:
import csv
import os

file_path = "Delivery_Logistics.csv"

# Check if the file exists before opening it
if os.path.exists(file_path):
  print("File found! Size (bytes):", os.path.getsize(file_path))

  # Read CSV headers and rows using core Python's csv module
  with open(file_path, mode="r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    headers = next(reader)  # First row is column names
    rows = list(reader)  # Remaining rows are records

  print("Columns:", len(headers), "| Rows:", len(rows))
  print("Sample record:", rows[0])

  # Write a simple text summary file using open(..., "w")
  with open("summary.txt", mode="w", encoding="utf-8") as out:
    out.write(f"Total Shipments: {len(rows)}\n")
    out.write(f"Total Columns: {len(headers)}\n")

  # Read back the saved text file to verify
  with open("summary.txt", mode="r", encoding="utf-8") as out:
    print("\nSaved Summary:\n" + out.read())
else:
  print("File not found. Please upload Delivery_Logistics.csv in Colab.")

File found! Size (bytes): 3710345
Columns: 15 | Rows: 25000
Sample record: ['250.99', 'delhivery', 'automobile parts', 'bike', 'same day', 'west', 'clear', '297.0', '46.96', '1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000008', 'no', 'delivered', '3', '1632.7205999999999']

Saved Summary:
Total Shipments: 25000
Total Columns: 15

